# E-commerce Product Recommender

Projet de recommandation produits e-commerce basé sur le collaborative filtering user-user.

Ce notebook :
- charge un sous-échantillon du dataset Amazon Electronics
- nettoie et filtre les données
- construit une matrice user-produit
- calcule la similarité entre utilisateurs
- génère des recommandations

In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## Chargement des données

In [15]:
df = pd.read_csv(
    "ratings_Electronics (1).csv",
    header=None,
    names=["user_id", "product_id", "rating", "timestamp"],
    usecols=[0, 1, 2],
    nrows=300000
)

print(df.head())
print(df.shape)
print("Nombre d'utilisateurs :", df["user_id"].nunique())
print("Nombre de produits :", df["product_id"].nunique())

          user_id  product_id  rating
0   AKM1MP6P0OYPR  0132793040     5.0
1  A2CX7LUOHB2NDG  0321732944     5.0
2  A2NWSAGRHCP8N5  0439886341     1.0
3  A2WNBOD3WNDNKT  0439886341     3.0
4  A1GI0U4ZRJA8WN  0439886341     1.0
(300000, 3)
Nombre d'utilisateurs : 252385
Nombre de produits : 18894


## Nettoyage du dataset

In [16]:
df = df.dropna().copy()
df["rating"] = df["rating"].astype(float)

print(df.isna().sum())
print(df["rating"].describe())

user_id       0
product_id    0
rating        0
dtype: int64
count    300000.000000
mean          3.997470
std           1.383179
min           1.000000
25%           3.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: rating, dtype: float64


## Filtrage des utilisateurs et produits peu fréquents

In [17]:
user_counts = df["user_id"].value_counts()
product_counts = df["product_id"].value_counts()

min_user_ratings = 3
min_product_ratings = 3

df_filtered = df[
    df["user_id"].isin(user_counts[user_counts >= min_user_ratings].index) &
    df["product_id"].isin(product_counts[product_counts >= min_product_ratings].index)
].copy()

print(df_filtered.shape)
print("Users after filtering :", df_filtered["user_id"].nunique())
print("Products after filtering :", df_filtered["product_id"].nunique())

(33885, 3)
Users after filtering : 8196
Products after filtering : 7332


## Sauvegarde du dataset filtré

In [18]:
df_filtered.to_csv("electronics_filtered.csv", index=False)
print("Fichier sauvegardé : electronics_filtered.csv")

Fichier sauvegardé : electronics_filtered.csv


## Matrice user-produit

In [19]:
user_item_matrix = df_filtered.pivot_table(
    index="user_id",
    columns="product_id",
    values="rating"
).fillna(0)

print(user_item_matrix.shape)
user_item_matrix.head()

(8196, 7332)


product_id,0528881469,0594451647,0594481813,0970407998,0972683275,1400501466,1400501520,1400501776,1400532620,1400532655,...,B00009V7MU,B00009V7MV,B00009VQA2,B00009VQE5,B00009VQG6,B00009VQJ7,B00009VQJF,B00009VQJR,B00009VQJZ,B00009VS6P
user_id,,,,,,,,,,,,,,,,,,,,,
A100NGGXRQF0AQ,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A100UD67AHFODS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A100WO06OQR8BQ,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A1016Q5UDME15Z,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A101AENIWYBBS0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Similarité entre utilisateurs

In [20]:
user_similarity = cosine_similarity(user_item_matrix)
print(user_similarity.shape)

(8196, 8196)


## Fonction de recommandation

In [21]:
def recommend_products(user_id, user_item_matrix, user_similarity, top_n=5):
    if user_id not in user_item_matrix.index:
        return []

    user_idx = user_item_matrix.index.get_loc(user_id)
    sim_scores = list(enumerate(user_similarity[user_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    similar_users = [i for i, score in sim_scores[1:11]]

    seen_products = set(
        user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    )

    product_scores = {}

    for sim_user_idx in similar_users:
        sim_user_id = user_item_matrix.index[sim_user_idx]
        sim_user_ratings = user_item_matrix.loc[sim_user_id]

        for product_id, rating in sim_user_ratings.items():
            if rating > 0 and product_id not in seen_products:
                product_scores[product_id] = product_scores.get(product_id, 0) + rating

    recommended = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)

    return [product_id for product_id, score in recommended[:top_n]]

## Statistiques produits

In [22]:
product_stats = df_filtered.groupby("product_id").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

product_stats = product_stats.sort_values(
    by=["rating_count", "avg_rating"],
    ascending=[False, False]
)

product_stats.head()

,product_id,avg_rating,rating_count
5604,B00007E7JU,4.542857,350
1875,B00004ZCJE,4.250765,327
5761,B00007KDVI,3.900990,202
6878,B00009R6TA,4.432990,194
939,B00004SB92,4.390625,192


## Fonction enrichie avec stats

In [23]:
def recommend_products_with_stats(user_id, user_item_matrix, user_similarity, df_filtered, top_n=5):
    recommendations = recommend_products(user_id, user_item_matrix, user_similarity, top_n=top_n)

    if not recommendations:
        return pd.DataFrame()

    stats = df_filtered.groupby("product_id").agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    ).reset_index()

    result = stats[stats["product_id"].isin(recommendations)].copy()
    return result.sort_values(by=["rating_count", "avg_rating"], ascending=[False, False])

## Test sur un utilisateur

In [24]:
test_user = user_item_matrix.index[0]

result = recommend_products_with_stats(
    test_user,
    user_item_matrix,
    user_similarity,
    df_filtered,
    top_n=5
)

print("Utilisateur testé :", test_user)
result

Utilisateur testé : A100NGGXRQF0AQ


,product_id,avg_rating,rating_count
5761,B00007KDVI,3.900990,202
5763,B00007KDVK,4.087719,57
4278,B000068UY6,3.959184,49
2708,B00005LENO,4.580645,31
2398,B00005AR4L,4.105263,19


In [27]:
def debug_user(user_id):
    print("=== PRODUITS DÉJÀ NOTÉS ===")
    seen = user_item_matrix.loc[user_id]
    seen = seen[seen > 0].sort_values(ascending=False)
    print(seen.head(10))

    print("\n=== RECOMMANDATIONS ===")
    recs = recommend_products_with_stats(
        user_id,
        user_item_matrix,
        user_similarity,
        df_filtered,
        top_n=5
    )
    display(recs)


# Test avec un utilisateur
test_user = user_item_matrix.index[0]
debug_user(test_user)

=== PRODUITS DÉJÀ NOTÉS ===
product_id
B00006B8C0    5.0
B00008IP5F    5.0
B000063Y9L    3.0
Name: A100NGGXRQF0AQ, dtype: float64

=== RECOMMANDATIONS ===


,product_id,avg_rating,rating_count
5761,B00007KDVI,3.900990,202
5763,B00007KDVK,4.087719,57
4278,B000068UY6,3.959184,49
2708,B00005LENO,4.580645,31
2398,B00005AR4L,4.105263,19


## Produits déjà notés

In [25]:
already_seen = user_item_matrix.loc[test_user]
already_seen = already_seen[already_seen > 0].sort_values(ascending=False)

print("Produits déjà notés par l'utilisateur :")
already_seen.head(10)

Produits déjà notés par l'utilisateur :


product_id
B00006B8C0    5.0
B00008IP5F    5.0
B000063Y9L    3.0
Name: A100NGGXRQF0AQ, dtype: float64

## Conclusion

In [26]:
print("Pipeline terminé avec succès.")
print("Le moteur recommande des produits à partir de profils utilisateurs similaires.")
print("Version future : ajout des métadonnées produits et d'un système hybride.")

Pipeline terminé avec succès.
Le moteur recommande des produits à partir de profils utilisateurs similaires.
Version future : ajout des métadonnées produits et d'un système hybride.
